# Phase 2: Preprocessing IPS (Revisi v2)
Melakukan cleaning dan pivot data menjadi format wide (snapshot) dengan penanganan `Age` yang dimodifikasi. 
Daripada membuang baris yang salah satu beacon-nya timeout (stale), kita menggantinya dengan nilai `-100` (Out of Range).

In [ ]:
import pandas as pd
import numpy as np

file_path = r'../Main_Data/combined_dataset_v6.csv'
df = pd.read_csv(file_path, sep=';')
print(f"Total data awal: {len(df)}")

In [ ]:
# Hapus baris Advertising Interval & RSSI_val NaN
df = df[df['Source'] != 'Advertising Interval'].copy()
df['RSSI_val'] = pd.to_numeric(df['RSSI_val'], errors='coerce')
df = df.dropna(subset=['RSSI_val']).copy()
df = df.drop(columns=['Source'])
print(f"Sisa dataset bersih: {len(df)} baris")

In [ ]:
# Urutkan berdasarkan waktu
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='mixed')
df = df.sort_values(['Table', 'Side', 'Timestamp']).reset_index(drop=True)

BEACONS = [3, 8, 33, 38]

In [ ]:
def build_snapshot_with_age(group):
    group = group.sort_values('Timestamp')
    pivot = group.pivot_table(index='Timestamp', columns='Beacon', values='RSSI_val', aggfunc='mean')
    pivot = pivot.reindex(columns=BEACONS)

    idx = pivot.index.to_series(name='Timestamp')
    value = pivot.ffill()

    age = pd.DataFrame(index=pivot.index)
    for b in BEACONS:
        last_seen_time = idx.where(pivot[b].notna()).ffill()
        age[b] = (idx - last_seen_time).dt.total_seconds()

    value.columns = [f'RSSI_{int(c)}' for c in value.columns]
    age.columns = [f'Age_{int(c)}' for c in age.columns]
    return pd.concat([value, age], axis=1)

snapshot = (
    df.groupby(['Table', 'Side'])
      .apply(build_snapshot_with_age, include_groups=False)
      .reset_index()
)
print(f"Total baris snapshot: {len(snapshot)}")

## Penanganan Nilai Basi (Stale Values)
Kita set `MAX_AGE = 5.0` detik. Jika umur suatu bacaan RSSI melebihi ini, maka artinya beacon tersebut sedang lost contact. 
Daripada membuang baris snapshot sepenuhnya (`dropna`), kita asumsikan sinyal tidak terdeteksi dan diisi dengan `-100`.

In [ ]:
MAX_AGE = 5.0
OUT_OF_RANGE_RSSI = -100.0

wide = snapshot.copy()
beacon_cols = [f'RSSI_{b}' for b in BEACONS]
age_cols = [f'Age_{b}' for b in BEACONS]

for b in BEACONS:
    rc, ac = f'RSSI_{b}', f'Age_{b}'
    # Jika belum ada pembacaan sama sekali (NaN) atau usianya terlalu tua (> MAX_AGE)
    invalid_mask = wide[ac].isna() | (wide[ac] > MAX_AGE)
    wide.loc[invalid_mask, rc] = OUT_OF_RANGE_RSSI

wide['Sample'] = wide.groupby(['Table', 'Side']).cumcount()
print(f"MAX_AGE = {MAX_AGE}s, OUT_OF_RANGE_RSSI = {OUT_OF_RANGE_RSSI}")

In [ ]:
# Kita hanya buang baris di mana KEEMPAT beacon nilainya -100 (artinya benar-benar kosong/warm-up)
valid_rows_mask = (wide[beacon_cols] > -100.0).any(axis=1)
wide_complete = wide[valid_rows_mask].reset_index(drop=True)

print(f"Total baris valid: {len(wide_complete)}")
print("Distribusi per Table & Side:\n", wide_complete.groupby(['Table', 'Side']).size().describe())
print("\nCek jumlah kombinasi Table dan Side yang valid:", len(wide_complete.groupby(['Table', 'Side'])))

In [ ]:
import os
output_dir = '../Main_Data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
output_name = os.path.join(output_dir, 'ips_dataset_wide_windowed_v2.csv')

wide_complete.to_csv(output_name, index=False)
print(f"Tersimpan: {output_name} ({wide_complete.shape[0]} baris)")